# Private vs PSU Banks in India — A Strategic Analysis (FY20–FY25)

*Author: Priyanshu Moudgil · BBA · Business Analyst Portfolio*

**Business question**: *You are advising the CEO of a major Indian PSU bank.
The board has asked — why are we losing ground to private banks year after year,
and what 3 things must change in the next 24 months?*

**Banks analyzed (6 banks × 6 years = 36 observations)**

| Private (winners) | PSU (laggards we're advising) |
|---|---|
| HDFC Bank | State Bank of India (SBI) |
| ICICI Bank | Punjab National Bank (PNB) |
| Axis Bank | Bank of Baroda (BOB) |

These 6 banks hold roughly 60–70% of all Indian banking assets.

**Metrics**

- **Total assets, deposits, advances, net profit** (in ₹ crore)
- **Gross NPA %** — loan-book quality (lower = better)
- **ROA %** — return on assets (higher = better)
- **NIM %** — net interest margin / pricing power (higher = better)
- **CASA %** — share of cheap current+savings deposits (higher = better)


## 1. Setup

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path("../data/bank_financials.csv")
DB   = Path("../data/bank_financials.db")
OUT  = Path("../outputs")
OUT.mkdir(exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams.update({"axes.titleweight": "bold", "figure.dpi": 110})

df = pd.read_csv(DATA)
df["financial_year"] = pd.Categorical(
    df["financial_year"],
    categories=["FY20","FY21","FY22","FY23","FY24","FY25"], ordered=True,
)
print("rows:", len(df), "| banks:", df['bank_name'].nunique(), "| years:", df['financial_year'].nunique())
df.head()

rows: 36 | banks: 6 | years: 6


,bank_name,bank_type,financial_year,total_assets_cr,deposits_cr,advances_cr,net_profit_cr,gross_npa_pct,roa_pct,nim_pct,casa_pct
0,HDFC Bank,Private,FY20,1530511,1147502,993703,26257,1.26,1.89,4.2,42.2
1,HDFC Bank,Private,FY21,1746870,1335060,1132837,31116,1.32,1.97,4.1,46.1
2,HDFC Bank,Private,FY22,2068535,1559217,1368821,36961,1.17,2.03,4.0,48.2
3,HDFC Bank,Private,FY23,2466081,1883394,1600586,44109,1.12,2.07,4.1,44.4
4,HDFC Bank,Private,FY24,4030190,2376888,2571920,64062,1.24,1.59,3.6,38.2


## 2. Data quality check

In [2]:
print("Missing values:")
print(df.isna().sum())
print()
print("Per-bank row counts (should all be 6):")
print(df.groupby('bank_name').size())
print()
print("Summary stats:")
df.describe().round(2)

Missing values:
bank_name          0
bank_type          0
financial_year     0
total_assets_cr    0
deposits_cr        0
advances_cr        0
net_profit_cr      0
gross_npa_pct      0
roa_pct            0
nim_pct            0
casa_pct           0
dtype: int64

Per-bank row counts (should all be 6):
bank_name
Axis Bank         6
Bank of Baroda    6
HDFC Bank         6
ICICI Bank        6
PNB               6
SBI               6
dtype: int64

Summary stats:


,total_assets_cr,deposits_cr,advances_cr,net_profit_cr,gross_npa_pct,roa_pct,nim_pct,casa_pct
count,36.00,36.00,36.00,36.00,36.00,36.00,36.00,36.00
mean,2243147.75,1754937.81,1383808.22,24146.50,4.41,1.11,3.48,42.91
std,1600839.46,1256738.80,956184.36,20668.15,3.56,0.73,0.52,3.24
min,830657.00,640105.00,471827.00,336.00,1.12,0.04,2.59,34.90
25%,1274069.25,1026203.50,732343.00,7766.25,1.79,0.53,3.08,40.83
50%,1548173.00,1242251.50,1006670.50,19099.50,3.26,1.04,3.42,42.60
75%,2167921.50,1678609.50,1426762.25,33162.25,5.58,1.83,3.96,45.35
max,6606290.00,5382190.00,4221000.00,70901.00,14.21,2.46,4.48,48.90


## 3. Finding #1 — The profitability gap held at ~1 percentage point

ROA (Return on Assets) is the cleanest single profitability metric — net profit divided by total assets.
A 1pp gap may sound small, but on a ₹50 lakh crore balance sheet that's ₹50,000 crore of foregone profit.


In [3]:
roa = df.groupby(['financial_year','bank_type'], observed=True)['roa_pct'].mean().unstack()
roa['gap_pp'] = roa['Private'] - roa['PSU']
roa.round(2)

bank_type,PSU,Private,gap_pp
financial_year,,,
FY20,0.16,0.97,0.81
FY21,0.23,1.36,1.13
FY22,0.51,1.69,1.18
FY23,0.72,1.67,0.95
FY24,0.92,1.93,1.01
FY25,1.12,1.98,0.86


**Insight.** Private ROA roughly doubled (0.97% → 1.98%). PSU ROA grew ~7x but from a tiny base (0.16% → 1.12%).
The gap has held at ~1pp across the cycle. PSUs *are* catching up — but slowly, and only because their
NPA clean-up is finally feeding through to the bottom line.


## 4. Finding #2 — The PSU NPA clean-up is real (and almost over)

This is the most flattering story PSU banks can tell. Six years ago, every ₹100 of PSU loans had
₹10 sitting in non-performing buckets. Today that figure is below ₹3.


In [4]:
gnpa = df.groupby(['financial_year','bank_type'], observed=True)['gross_npa_pct'].mean().unstack()
gnpa['psu_excess_pp'] = gnpa['PSU'] - gnpa['Private']
gnpa.round(2)

bank_type,PSU,Private,psu_excess_pp
financial_year,,,
FY20,9.92,3.88,6.04
FY21,9.32,3.33,6.00
FY22,7.45,2.53,4.92
FY23,5.10,1.98,3.12
FY24,3.63,1.68,1.95
FY25,2.68,1.44,1.23


**Insight.** The PSU "excess NPA" gap narrowed from **6.04pp (FY20) → 1.23pp (FY25)** — a ~5pp
improvement. The IBC, write-offs, and credit cycle did most of the work. **But** the easy gains are gone;
from here, PSU asset quality has to be earned through better underwriting, not bankruptcy court.


## 5. Finding #3 — Growth: PSUs aren't shrinking, they're just smaller

In [5]:
adv = df.pivot(index='bank_name', columns='financial_year', values='advances_cr')
adv['cagr_pct'] = ((adv['FY25'] / adv['FY20']) ** (1/5) - 1) * 100
adv = adv.join(df[['bank_name','bank_type']].drop_duplicates().set_index('bank_name'))
adv[['FY20','FY25','cagr_pct','bank_type']].sort_values('cagr_pct', ascending=False).round(1)

,FY20,FY25,cagr_pct,bank_type
bank_name,,,,
HDFC Bank,993703,2724940,22.4,Private
PNB,471827,1116000,18.8,PSU
ICICI Bank,645289,1341766,15.8,Private
Axis Bank,571424,1040810,12.7,Private
SBI,2325290,4221000,12.7,PSU
Bank of Baroda,690121,1230000,12.3,PSU


**Insight.** Loan-book CAGR over FY20–FY25:

- HDFC **22.4%** — but inflated by the FY24 HDFC Ltd merger; underlying organic growth is ~14–15%
- PNB **18.8%**, ICICI **15.8%** — PSUs aren't growth-laggards, they're growing at private-bank pace
- Axis = SBI = **12.7%** — large-bank base effects

The narrative that "PSUs are dying" is wrong. They're growing the loan book — they just earn less on it.


## 6. Finding #4 — The CASA advantage just evaporated

CASA = Current + Savings Accounts. It's the cheapest funding a bank has — current accounts pay 0% interest,
savings 3–4%. Private banks have always claimed a structural CASA edge. The data says: not anymore.


In [6]:
casa = df.groupby(['financial_year','bank_type'], observed=True)['casa_pct'].mean().unstack()
casa['private_advantage_pp'] = casa['Private'] - casa['PSU']
casa.round(2)

bank_type,PSU,Private,private_advantage_pp
financial_year,,,
FY20,42.33,42.83,0.50
FY21,44.50,45.80,1.30
FY22,44.93,47.37,2.43
FY23,42.37,45.73,3.37
FY24,40.67,40.73,0.07
FY25,39.57,38.10,-1.47


**Insight.** Private CASA peaked at 47.4% in FY22 and has fallen to **38.1%** in FY25.
PSU CASA also fell, but less violently. For the first time in the dataset, in FY25 **PSUs (39.6%) actually
beat private banks (38.1%) on CASA**. The reason: customers moved cash into 7–8% fixed deposits as
interest rates rose. Private banks lost their structural funding advantage — at least for now.

**This is the surprise of the analysis.** It says PSU CEOs shouldn't waste capital chasing CASA share —
the war is over and the rate cycle decided it.


## 7. Finding #5 — NIM: the persistent ~90bps spread

In [7]:
nim_by_bank = df.groupby(['bank_name','bank_type'], observed=True)['nim_pct'].mean().reset_index()
nim_by_bank.sort_values(['bank_type','nim_pct'], ascending=[True, False]).round(2)

,bank_name,bank_type,nim_pct
5,SBI,PSU,3.19
1,Bank of Baroda,PSU,3.01
4,PNB,PSU,2.91
3,ICICI Bank,Private,4.06
2,HDFC Bank,Private,3.98
0,Axis Bank,Private,3.74


In [8]:
nim_by_type = df.groupby('bank_type')['nim_pct'].mean()
print(f"Avg private NIM: {nim_by_type['Private']:.2f}%")
print(f"Avg PSU     NIM: {nim_by_type['PSU']:.2f}%")
print(f"Spread:           {nim_by_type['Private'] - nim_by_type['PSU']:.2f}pp")

Avg private NIM: 3.93%
Avg PSU     NIM: 3.04%
Spread:           0.89pp


**Insight.** Private banks earn ~90bps more on every rupee they lend. This isn't a CASA story
(see Finding #4 — CASA gap is gone). It is a **pricing-power and asset-mix story**: private banks
lend more to retail and unsecured segments where yields are 200–300bps higher than corporate.
NIM is the single best predictor of long-run ROA, and it explains most of the residual profitability gap.


## 8. FY25 league table — synthesizing everything

Rank every bank on each KPI, then average the ranks for a single bottom-line ordering.


In [9]:
fy25 = df[df['financial_year']=='FY25'].copy()
fy25['rk_npa']    = fy25['gross_npa_pct'].rank(ascending=True)
fy25['rk_roa']    = fy25['roa_pct'].rank(ascending=False)
fy25['rk_nim']    = fy25['nim_pct'].rank(ascending=False)
fy25['rk_casa']   = fy25['casa_pct'].rank(ascending=False)
fy25['rk_profit'] = fy25['net_profit_cr'].rank(ascending=False)
fy25['avg_rank']  = fy25[['rk_npa','rk_roa','rk_nim','rk_casa','rk_profit']].mean(axis=1)
cols = ['bank_name','bank_type','total_assets_cr','net_profit_cr','roa_pct',
        'gross_npa_pct','nim_pct','casa_pct','avg_rank']
fy25[cols].sort_values('avg_rank').round(2)

,bank_name,bank_type,total_assets_cr,net_profit_cr,roa_pct,gross_npa_pct,nim_pct,casa_pct,avg_rank
17,Axis Bank,Private,1610327,28055,1.88,1.30,3.98,41.00,2.0
11,ICICI Bank,Private,1918233,47227,2.46,1.70,4.40,38.40,2.6
5,HDFC Bank,Private,4392420,70792,1.61,1.33,3.90,34.90,3.2
23,SBI,PSU,6606290,70901,1.10,1.82,3.09,39.97,3.2
35,Bank of Baroda,PSU,1710000,20716,1.26,2.26,3.02,39.95,4.4
29,PNB,PSU,1650000,16630,1.00,3.95,2.93,38.80,5.6


**Take-away.** All three private banks lead. SBI ties HDFC on combined ranks — its NPA clean-up
and sheer scale make it the only PSU that competes with the top private banks on every dimension.
PNB still occupies the bottom.


## 9. Strategic recommendations

Three things the PSU CEO should commit to in the next 24 months.

### 1. **Stop chasing CASA. Chase NIM.**
The data is brutal: private CASA fell faster than PSU CASA. The funding war is over.
The real war is on the **asset** side — pricing, retail mix, and digital underwriting.
*Action*: shift the loan book mix by 5pp from corporate to high-yield retail (personal,
credit card, small-business) over 8 quarters. Stand up a digital-lending JV with a fintech
to underwrite at scale.

### 2. **Defend the NPA gains by re-engineering underwriting, not by lending less.**
The IBC won the first NPA war. The second will be won by data: bureau scores, GST flows,
real-time monitoring. *Action*: cap manual exception approvals at 10% of new sanctions,
mandate model-driven decisioning for every loan under ₹5 crore by Q4 FY27, and tie 30% of
relationship-manager variable pay to vintage NPA performance, not disbursal volume.

### 3. **Become the bank for Bharat's middle 60%.**
Private banks own urban affluent; fintechs own digital natives. The 600M middle-income,
semi-urban, GST-registered SMB segment is largely under-served. PSUs have the branch network
and trust — they just don't have the digital onboarding. *Action*: rebuild mobile + chat
account opening to a 90-second flow, partner with two regional payment-aggregators for
SMB collections, and target ₹5 lakh crore of new SMB advances by FY28.

### What this means for fintechs

A re-rated PSU is the most underpriced distribution channel in India. The bank with 23,000
branches and 50 crore accounts has 90% of the customer file and 10% of the digital UX.
A fintech that productizes underwriting, KYC, or collections **as infrastructure that SBI/PNB/BOB
can rent** captures the upside without funding the balance sheet. The winners of the next
decade in BFSI may not be the bank or the fintech — they'll be the picks-and-shovels providers
sitting between them.
